#  Machine Learning Model Training & Evaluation

##  Objective
Train, evaluate, and compare Gradient Boosting (**XGBoost**, **LightGBM**), **Random Forest**, and **Ridge Regressor** models to predict daily state-level Aadhaar enrolments using engineered temporal, lag, rolling, and interaction features.

In [1]:
import os
import glob
import json
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb

# Inline Data Ingestion & State Normalization Utilities
import os
import glob
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

CANONICAL_STATES = [
    "Andaman & Nicobar", "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar",
    "Chandigarh", "Chhattisgarh", "Dadra & Nagar Haveli and Daman & Diu", "Delhi",
    "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jammu and Kashmir", "Jharkhand",
    "Karnataka", "Kerala", "Ladakh", "Lakshadweep", "Madhya Pradesh", "Maharashtra",
    "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Puducherry", "Punjab",
    "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh",
    "Uttarakhand", "West Bengal"
]

STATE_ALIASES = {
    "andaman and nicobar islands": "Andaman & Nicobar",
    "andaman & nicobar islands": "Andaman & Nicobar",
    "a & n islands": "Andaman & Nicobar",
    "andhra pradesh": "Andhra Pradesh",
    "arunachal pradesh": "Arunachal Pradesh",
    "assam": "Assam",
    "bihar": "Bihar",
    "chandigarh": "Chandigarh",
    "chhattisgarh": "Chhattisgarh",
    "chhatisgarh": "Chhattisgarh",
    "dadra and nagar haveli": "Dadra & Nagar Haveli and Daman & Diu",
    "daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "dadra and nagar haveli and daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "delhi": "Delhi",
    "nct of delhi": "Delhi",
    "goa": "Goa",
    "gujarat": "Gujarat",
    "haryana": "Haryana",
    "himachal pradesh": "Himachal Pradesh",
    "jammu and kashmir": "Jammu and Kashmir",
    "jammu & kashmir": "Jammu and Kashmir",
    "jharkhand": "Jharkhand",
    "karnataka": "Karnataka",
    "kerala": "Kerala",
    "ladakh": "Ladakh",
    "lakshadweep": "Lakshadweep",
    "madhya pradesh": "Madhya Pradesh",
    "maharashtra": "Maharashtra",
    "manipur": "Manipur",
    "meghalaya": "Meghalaya",
    "mizoram": "Mizoram",
    "nagaland": "Nagaland",
    "odisha": "Odisha",
    "orissa": "Odisha",
    "puducherry": "Puducherry",
    "pondicherry": "Puducherry",
    "punjab": "Punjab",
    "rajasthan": "Rajasthan",
    "sikkim": "Sikkim",
    "tamil nadu": "Tamil Nadu",
    "telangana": "Telangana",
    "tripura": "Tripura",
    "uttar pradesh": "Uttar Pradesh",
    "uttarakhand": "Uttarakhand",
    "uttaranchal": "Uttarakhand",
    "west bengal": "West Bengal"
}

def _normalize_state_name(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()
    return STATE_ALIASES.get(val_str, str(val).strip().title())

def load_and_preprocess_raw_data(data_dir="."):
    enrol_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_enrolment/**/*.csv'), recursive=True))
    enrol_list = []
    for f in enrol_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        enrol_list.append(df)
    
    enrol_df = pd.concat(enrol_list, ignore_index=True) if enrol_list else pd.DataFrame()
        
    if not enrol_df.empty:
        enrol_df['date'] = pd.to_datetime(enrol_df['date'], format='%d-%m-%Y', errors='coerce')
        enrol_df['norm_state'] = enrol_df['state'].apply(_normalize_state_name)
        enrol_df['total_enrolments'] = enrol_df['age_0_5'].fillna(0) + enrol_df['age_5_17'].fillna(0) + enrol_df['age_18_greater'].fillna(0)
        enrol_daily = enrol_df.groupby(['date', 'norm_state'])[['age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments']].sum().reset_index()
    else:
        enrol_daily = pd.DataFrame(columns=['date', 'norm_state', 'age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments'])

    demo_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_demographic/**/*.csv'), recursive=True))
    demo_list = []
    for f in demo_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        demo_list.append(df)
    
    if demo_list:
        demo_df = pd.concat(demo_list, ignore_index=True)
        demo_df['date'] = pd.to_datetime(demo_df['date'], format='%d-%m-%Y', errors='coerce')
        demo_df['norm_state'] = demo_df['state'].apply(_normalize_state_name)
        demo_df['demo_total'] = demo_df['demo_age_5_17'].fillna(0) + demo_df['demo_age_17_'].fillna(0)
        demo_daily = demo_df.groupby(['date', 'norm_state'])[['demo_age_5_17', 'demo_age_17_', 'demo_total']].sum().reset_index()
    else:
        demo_daily = pd.DataFrame(columns=['date', 'norm_state', 'demo_age_5_17', 'demo_age_17_', 'demo_total'])

    bio_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_biometric/**/*.csv'), recursive=True))
    bio_list = []
    for f in bio_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        bio_list.append(df)
        
    if bio_list:
        bio_df = pd.concat(bio_list, ignore_index=True)
        bio_df['date'] = pd.to_datetime(bio_df['date'], format='%d-%m-%Y', errors='coerce')
        bio_df['norm_state'] = bio_df['state'].apply(_normalize_state_name)
        bio_df['bio_total'] = bio_df['bio_age_5_17'].fillna(0) + bio_df['bio_age_17_'].fillna(0)
        bio_daily = bio_df.groupby(['date', 'norm_state'])[['bio_age_5_17', 'bio_age_17_', 'bio_total']].sum().reset_index()
    else:
        bio_daily = pd.DataFrame(columns=['date', 'norm_state', 'bio_age_5_17', 'bio_age_17_', 'bio_total'])

    merged = pd.merge(enrol_daily, demo_daily, on=['date', 'norm_state'], how='outer')
    merged = pd.merge(merged, bio_daily, on=['date', 'norm_state'], how='outer')
    
    merged['total_enrolments'] = merged['total_enrolments'].fillna(0)
    merged['demo_total'] = merged['demo_total'].fillna(0)
    merged['bio_total'] = merged['bio_total'].fillna(0)

    return merged


def create_feature_pipeline(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['norm_state', 'date']).reset_index(drop=True)
    
    states = df['norm_state'].dropna().unique()
    all_dates = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
    
    grid = pd.MultiIndex.from_product([states, all_dates], names=['norm_state', 'date']).to_frame().reset_index(drop=True)
    full_df = pd.merge(grid, df, on=['norm_state', 'date'], how='left')
    
    num_cols = ['age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments',
                'demo_age_5_17', 'demo_age_17_', 'demo_total',
                'bio_age_5_17', 'bio_age_17_', 'bio_total']
    for col in num_cols:
        if col in full_df.columns:
            full_df[col] = full_df[col].fillna(0)
            
    full_df['day_of_week'] = full_df['date'].dt.dayofweek
    full_df['day_of_month'] = full_df['date'].dt.day
    full_df['month'] = full_df['date'].dt.month
    full_df['quarter'] = full_df['date'].dt.quarter
    full_df['day_of_year'] = full_df['date'].dt.dayofyear
    full_df['is_weekend'] = full_df['day_of_week'].isin([5, 6]).astype(int)
    
    full_df['sin_day_of_week'] = np.sin(2 * np.pi * full_df['day_of_week'] / 7)
    full_df['cos_day_of_week'] = np.cos(2 * np.pi * full_df['day_of_week'] / 7)
    full_df['sin_month'] = np.sin(2 * np.pi * full_df['month'] / 12)
    full_df['cos_month'] = np.cos(2 * np.pi * full_df['month'] / 12)

    full_df['state_cat'] = full_df['norm_state'].astype('category').cat.codes

    feature_dfs = []
    for state, group in full_df.groupby('norm_state'):
        group = group.sort_values('date').copy()
        
        for lag in [1, 7, 14, 30]:
            group[f'lag_{lag}'] = group['total_enrolments'].shift(lag)
            group[f'bio_lag_{lag}'] = group['bio_total'].shift(lag)
            group[f'demo_lag_{lag}'] = group['demo_total'].shift(lag)
            
        for window in [7, 14, 30]:
            group[f'rolling_mean_{window}'] = group['total_enrolments'].shift(1).rolling(window=window, min_periods=1).mean()
            group[f'rolling_std_{window}'] = group['total_enrolments'].shift(1).rolling(window=window, min_periods=1).std().fillna(0)
            group[f'bio_rolling_mean_{window}'] = group['bio_total'].shift(1).rolling(window=window, min_periods=1).mean()
            group[f'demo_rolling_mean_{window}'] = group['demo_total'].shift(1).rolling(window=window, min_periods=1).mean()

        group['bio_to_enrol_ratio'] = (group['bio_rolling_mean_7'] / (group['rolling_mean_7'] + 1)).fillna(0)
        group['demo_to_enrol_ratio'] = (group['demo_rolling_mean_7'] / (group['rolling_mean_7'] + 1)).fillna(0)
        
        feature_dfs.append(group)
        
    processed_df = pd.concat(feature_dfs, ignore_index=True)
    processed_df = processed_df.fillna(0)
    
    return processed_df


# 1. Load panel dataset & run feature engineering pipeline
raw_panel = load_and_preprocess_raw_data(data_dir=".")
processed_df = create_feature_pipeline(raw_panel)

print(f"Dataset shape: {processed_df.shape}")
print(f"Features: {processed_df.columns.tolist()}")


Dataset shape: (15912, 49)
Features: ['norm_state', 'date', 'age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments', 'demo_age_5_17', 'demo_age_17_', 'demo_total', 'bio_age_5_17', 'bio_age_17_', 'bio_total', 'day_of_week', 'day_of_month', 'month', 'quarter', 'day_of_year', 'is_weekend', 'sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month', 'state_cat', 'lag_1', 'bio_lag_1', 'demo_lag_1', 'lag_7', 'bio_lag_7', 'demo_lag_7', 'lag_14', 'bio_lag_14', 'demo_lag_14', 'lag_30', 'bio_lag_30', 'demo_lag_30', 'rolling_mean_7', 'rolling_std_7', 'bio_rolling_mean_7', 'demo_rolling_mean_7', 'rolling_mean_14', 'rolling_std_14', 'bio_rolling_mean_14', 'demo_rolling_mean_14', 'rolling_mean_30', 'rolling_std_30', 'bio_rolling_mean_30', 'demo_rolling_mean_30', 'bio_to_enrol_ratio', 'demo_to_enrol_ratio']


## ⚙️ Model Training & Chronological Evaluation

We split data chronologically into an 80% historical training set and a 20% recent testing set.
We train XGBoost, LightGBM, Random Forest, and Ridge models and compare their metrics (R², RMSE, MAE).

In [2]:
target_col = 'total_enrolments'
exclude_cols = [
    'date', 'norm_state', 'state', 'district', 'pincode',
    'age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments',
    'demo_age_5_17', 'demo_age_17_', 'demo_total',
    'bio_age_5_17', 'bio_age_17_', 'bio_total'
]
feature_cols = [c for c in processed_df.columns if c not in exclude_cols]

# Temporal split (80/20)
unique_dates = sorted(processed_df['date'].unique())
split_date = unique_dates[int(len(unique_dates) * 0.8)]

train_df = processed_df[processed_df['date'] < split_date].copy()
test_df = processed_df[processed_df['date'] >= split_date].copy()

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

models = {
    'XGBoost': xgb.XGBRegressor(n_estimators=200, learning_rate=0.03, max_depth=5, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Ridge Baseline': Ridge(alpha=10.0)
}

os.makedirs('pkl_models', exist_ok=True)

results = []
preds_dict = {}
best_model_obj = None
best_r2 = -float('inf')

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = np.maximum(0, model.predict(X_test))
    preds_dict[name] = y_pred
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    filename = f"{name.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(model, os.path.join('pkl_models', filename))
    
    if r2 > best_r2:
        best_r2 = r2
        best_model_obj = model
    
    results.append({'Model': name, 'Test R²': round(r2, 4), 'RMSE': round(rmse, 2), 'MAE': round(mae, 2)})

if best_model_obj is not None:
    joblib.dump(best_model_obj, os.path.join('pkl_models', 'best_model.pkl'))

results_df = pd.DataFrame(results).sort_values('Test R²', ascending=False)
results_df


,Model,Test R²,RMSE,MAE
3,Ridge Baseline,0.3944,1422.31,619.18
0,XGBoost,-0.3642,2134.64,1096.22
1,LightGBM,-0.4978,2236.74,1173.43
2,Random Forest,-1.0490,2616.16,1117.38
